# Chequeo de disponibilidad de datos (WDI)


**Objetivo de este notebook:**
Antes de clasificar los 975 indicadores candidats (`Relevante` + `considerar` en
`df_clasificado.xlsx`) dentro de las cuatro dimensiones teóricas, se evalúa su
**disponibilidad real de datos**: cobertura temporal (años con dato) y cobertura entre
países (cuántos países soberanos reportan la serie). Esto permite descartar indicadores
inútiles por falta de datos antes de invertir tiempo en su clasificación teórica.

**Decisión metodológica clave:** en lugar de solicitar los 975 indicadores uno por uno vía
API (lento, frágil ante *rate limits*, difícil de auditar), se descarga **una sola vez**
el archivo bulk oficial del Banco Mundial (`WDI_csv.zip`), que contiene todos los
indicadores y todos los países/agregados. El filtrado y cómputo posterior se hacen
localmente en memoria con `pandas`.


# Sección 1: Datos completos del WDI (217 países)

## Bloque 1 — Importación de librerías

**Objetivo:** cargar las herramientas necesarias para descargar el archivo bulk del WDI,
descomprimirlo y organizar los resultados en tablas.

Se usa `requests` (descarga HTTP estándar, ya usado en el
notebook 01) y `zipfile`/`io` de la librería estándar para evitar dependencias adicionales
innecesarias. No se usa `wbgapi` en este paso porque ese paquete está pensado para
consultas puntuales indicador por indicador, exactamente el patrón que queremos evitar por
costo computacional/de red.



In [57]:
import requests
import zipfile
import io
from pathlib import Path

import pandas as pd
import numpy as np


## Librerías de visualización

Se importan acá, una única vez, para que estén disponibles en todo el resto del notebook sin repetir imports en cada bloque de gráficos.

In [58]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Acá se van acumulando todas las figuras (nombre, objeto fig) para el export final a HTML
figuras_reporte = []
tablas_reporte = []


## Bloque 2 — Descarga del archivo bulk del WDI

**Objetivo:** descargar una única vez el archivo `WDI_csv.zip`, que contiene la base
completa de indicadores del World Development Indicators (todos los países, todos los
años, todos los indicadores).

El archivo resultante queda guardado en disco con control de que no se vuelva a descargar si
ya existe, lo que hace el notebook reproducible y evita descargas repetidas accidentales.

**Resultado esperado:** un archivo `WDI_csv.zip` en la carpeta `data/raw/`, y un mensaje
con su tamaño en MB. Si `resp.status_code` es inferior a 200mb, revisar conexión a internet o si
cambió la URL oficial de descarga.


In [59]:
RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

WDI_ZIP_URL = "https://databankfiles.worldbank.org/public/ddpext_download/WDI_CSV.zip"
zip_path = RAW_DIR / "WDI_csv.zip"

if zip_path.exists():
    print(f"Ya existe {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB). No se vuelve a descargar.")
else:
    resp = requests.get(WDI_ZIP_URL, timeout=120)
    resp.raise_for_status()
    zip_path.write_bytes(resp.content)
    print(f"Descargado {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")


Ya existe data\raw\WDI_csv.zip (283.2 MB). No se vuelve a descargar.


## Bloque 3 — Extracción de los archivos relevantes del ZIP

**Objetivo:** extraer del ZIP únicamente los dos archivos que necesitamos: los datos
(`WDICSV.csv`) y la metadata de países (`WDICountry.csv`), sin descomprimir archivos que
no vamos a usar (notas de series, metadata de indicadores ya la tenemos en
`df_clasificado.xlsx`).

**Justificación metodológica:** `WDICountry.csv` es la fuente que usamos para distinguir
país soberano de agregado regional/de ingreso: el Banco Mundial
identifica los países soberanos porque tienen un valor no vacío en la columna `Region`;
los agregados (ej. "World", "Latin America & Caribbean", "OECD members") tienen `Region`
vacío aunque tengan un código de 3 letras igual que un país.

**Resultado esperado:** dos archivos CSV extraídos en `data/raw/`. Verificar que
`WDICountry.csv` tenga alrededor de 217 filas con `Region` no vacío.


In [60]:
with zipfile.ZipFile(zip_path) as z:
    names = z.namelist()
    data_file = [n for n in names if n.upper().endswith("WDICSV.CSV")][0]
    country_file = [n for n in names if n.upper().endswith("WDICOUNTRY.CSV")][0]
    z.extract(data_file, RAW_DIR)
    z.extract(country_file, RAW_DIR)

data_path = RAW_DIR / data_file
country_path = RAW_DIR / country_file
print(data_path, country_path)


data\raw\WDICSV.csv data\raw\WDICountry.csv


## Bloque 4 — Lista de países soberanos (exclusión de agregados)

**Objetivo:** construir la lista de códigos ISO3 de países, excluyendo
agregados regionales.
. 


**Resultado esperado:** una lista `paises_soberanos` con ~217 códigos ISO3. 


In [61]:
country_meta = pd.read_csv(country_path)

paises_soberanos = (
    country_meta.loc[country_meta["Region"].notna(), "Country Code"]
    .unique()
    .tolist()
)

print(f"Países soberanos identificados: {len(paises_soberanos)}")
assert "ARG" in paises_soberanos, "Argentina no está en la lista — revisar filtro."


Países soberanos identificados: 217


## Bloque 5 — Lista de indicadores candidatos (975)

**Objetivo:** extraer de `df_clasificado.xlsx` los códigos WDI de los indicadores
marcados como `Relevante` o `considerar`, que son el universo de 975 indicadores a
evaluar (decisión ya acordada en la Fase 1).

**Justificación metodológica:** se excluyen los `no_relevante` porque ya fueron
descartados por criterio teórico en el paso anterior; no tiene sentido gastar cómputo en
medir su disponibilidad.

**Resultado esperado:** una lista `indicadores_candidatos` de longitud 974 (905 + 69,
según el conteo real de tu archivo; el número "975" mencionado antes era aproximado).


In [62]:
df_clasificado = pd.read_excel("df_clasificado.xlsx")

categorias_incluidas = ["Relevante", "considerar"]
indicadores_candidatos = (
    df_clasificado.loc[
        df_clasificado["Relevancia_TFM"].isin(categorias_incluidas), "Código WDI"
    ]
    .unique()
    .tolist()
)

print(f"Indicadores candidatos: {len(indicadores_candidatos)}")


Indicadores candidatos: 980


## Bloque 6 — Carga y filtrado del bulk WDI

**Objetivo:** cargar `WDICSV.csv` y quedarnos únicamente con las filas de las unidades economicas 
(países, no agregaciones) y cuyo indicador esté en la lista de 974 candidatos.

**Resultado esperado:** un DataFrame `wdi` con como máximo 974 × 217 = 211.358 filas
(en la práctica menos, porque no todos los indicadores tienen fila para todos los
países). Verificar `wdi["Indicator Code"].nunique()` y `wdi["Country Code"].nunique()`.


In [63]:
wdi_raw = pd.read_csv(data_path, low_memory=False)

wdi = wdi_raw[
    wdi_raw["Country Code"].isin(paises_soberanos)
    & wdi_raw["Indicator Code"].isin(indicadores_candidatos)
].copy()

print(f"Filas: {len(wdi)}")
print(f"Indicadores presentes: {wdi['Indicator Code'].nunique()} / {len(indicadores_candidatos)}")
print(f"Países presentes: {wdi['Country Code'].nunique()} / {len(paises_soberanos)}")


Filas: 212660
Indicadores presentes: 980 / 980
Países presentes: 217 / 217


## Bloque 7 — Reshape de formato ancho a formato largo

**Objetivo:** transformar la tabla de formato ancho (una columna por año) a formato largo
(`Country Code`, `Indicator Code`, `Year`, `Value`), que es el formato necesario para
calcular métricas de cobertura de forma vectorizada.

**Justificación metodológica:** se conserva todo el rango de años que trae el archivo
(desde 1960) para luego evaluar cómo es la cobertura de datos en el tiempo. 
Calculando la cobertura año a año una sola vez, en el Bloque 9 vas a poder generar ambos recortes sin repetir este cómputo.

**Resultado esperado:** un DataFrame `wdi_long` en formato largo. Verificar
`wdi_long["Year"].min()` y `.max()` para confirmar el rango real que trae el archivo.


In [64]:
year_cols = [c for c in wdi.columns if c.strip().isdigit()]

wdi_long = wdi.melt(
    id_vars=["Country Code", "Indicator Code"],
    value_vars=year_cols,
    var_name="Year",
    value_name="Value",
)
wdi_long["Year"] = wdi_long["Year"].astype(int)

print(f"Rango de años disponible: {wdi_long['Year'].min()} - {wdi_long['Year'].max()}")
print(f"Filas en formato largo: {len(wdi_long)}")


Rango de años disponible: 1960 - 2025
Filas en formato largo: 14035560


## Bloque 8 — Métricas de disponibilidad

## Tabla disponibilidad según indicador

**Objetivo:** construir disponibilidad_indicador_full, con una fila por cada uno de los 974 indicadores candidatos, indicando qué % de países soberanos tienen al menos un dato en algún año, y el primer/último año en que ese indicador tiene dato — sin recortar por ventana temporal.


Resultado esperado: un DataFrame de 974 filas. primer_anio/ultimo_anio numéricos cuando hay dato, y el string "sin_dato" cuando el indicador no tiene ningún dato para ningún país soberano.

In [65]:
# Totales base (necesarios para todos los % de este notebook en adelante)
n_paises_total = wdi_long["Country Code"].nunique()
n_indicadores_total = len(indicadores_candidatos)

print(f"Países totales: {n_paises_total}")
print(f"Indicadores totales: {n_indicadores_total}")

Países totales: 217
Indicadores totales: 980


In [66]:
paises_con_dato_full = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Indicator Code")["Country Code"]
    .nunique()
    .rename("n_paises_con_dato")
)

rango_anios_full = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Indicator Code")["Year"]
    .agg(primer_anio="min", ultimo_anio="max")
)

disponibilidad_indicador_full = (
    pd.DataFrame(index=indicadores_candidatos)
    .join(paises_con_dato_full)
    .join(rango_anios_full)
)

disponibilidad_indicador_full["n_paises_con_dato"] = disponibilidad_indicador_full["n_paises_con_dato"].fillna(0)
disponibilidad_indicador_full["pct_paises_con_dato"] = (
    disponibilidad_indicador_full["n_paises_con_dato"] / n_paises_total
)

# Etiqueta categórica: los años nunca se usan aritméticamente, así que "sin_dato" es seguro acá
disponibilidad_indicador_full["primer_anio"] = disponibilidad_indicador_full["primer_anio"].fillna("sin_dato")
disponibilidad_indicador_full["ultimo_anio"] = disponibilidad_indicador_full["ultimo_anio"].fillna("sin_dato")

disponibilidad_indicador_full = disponibilidad_indicador_full.sort_values("pct_paises_con_dato")
disponibilidad_indicador_full.shape

(980, 4)

In [67]:
disponibilidad_indicador_full.head()

,n_paises_con_dato,primer_anio,ultimo_anio,pct_paises_con_dato
SM.POP.RRWA.EO,1,1960,2025,0.004608
SM.POP.OPIP.EO,2,2018,2025,0.009217
SM.POP.RRWA.EA,4,1960,2025,0.018433
SH.STA.FGMS.ZS,30,1990,2023,0.138249
SM.POP.OPIP.EA,31,2018,2025,0.142857


## Tabla disponibilidad según país
Acá quiero mostrar la disponibilidad de datos en CADA país. 
Cuántos indicadores tengo por país. Primer y último año con datos.
Qué porcentaje de indicadores hay país

In [68]:
indicadores_con_dato = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Country Code")["Indicator Code"]
    .nunique()
    .rename("n_indicadores_con_dato")
)

rango_anios_pais = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Country Code")["Year"]
    .agg(primer_anio="min", ultimo_anio="max")
)

disponibilidad_pais = (
    pd.DataFrame(index=paises_soberanos)
    .join(indicadores_con_dato)
    .join(rango_anios_pais)
)

disponibilidad_pais["n_indicadores_con_dato"] = disponibilidad_pais["n_indicadores_con_dato"].fillna(0)
disponibilidad_pais["pct_indicadores_con_dato"] = (
    disponibilidad_pais["n_indicadores_con_dato"] / n_indicadores_total
)

disponibilidad_pais["primer_anio"] = disponibilidad_pais["primer_anio"].fillna("sin_dato")
disponibilidad_pais["ultimo_anio"] = disponibilidad_pais["ultimo_anio"].fillna("sin_dato")

# Region es descriptiva: se une después de calcular todo, no interviene en el cálculo
region_por_pais = country_meta.set_index("Country Code")["Region"]
disponibilidad_pais["Region"] = disponibilidad_pais.index.map(region_por_pais)
disponibilidad_pais["Region"] = disponibilidad_pais["Region"].fillna("sin_dato")

disponibilidad_pais = disponibilidad_pais.sort_values("pct_indicadores_con_dato")
disponibilidad_pais.shape

(217, 5)

In [69]:
disponibilidad_pais.head()

,n_indicadores_con_dato,primer_anio,ultimo_anio,pct_indicadores_con_dato,Region
MAF,110,1960,2025,0.112245,Latin America & Caribbean
IMN,174,1960,2025,0.177551,Europe & Central Asia
CHI,184,1960,2025,0.187755,Europe & Central Asia
MNP,187,1960,2025,0.190816,East Asia & Pacific
GIB,231,1960,2025,0.235714,Europe & Central Asia


## Tabla de disponibilidad según año

In [70]:
anios_totales = sorted(wdi_long["Year"].unique())

paises_con_dato_anio = (
    wdi_long.dropna(subset=["Value"])
    .groupby("Year")["Country Code"]
    .nunique()
    .rename("n_paises_con_dato")
)

disponibilidad_anio = pd.DataFrame(index=anios_totales).join(paises_con_dato_anio)
disponibilidad_anio["n_paises_con_dato"] = disponibilidad_anio["n_paises_con_dato"].fillna(0)
disponibilidad_anio["pct_paises_con_dato"] = disponibilidad_anio["n_paises_con_dato"] / n_paises_total

# Unión con Region solo para este cálculo (no se modifica wdi_long)
wdi_long_region = wdi_long.merge(
    country_meta[["Country Code", "Region"]], on="Country Code", how="left"
)

n_paises_por_region = (
    country_meta.loc[country_meta["Region"].notna()]
    .groupby("Region")["Country Code"]
    .nunique()
)

paises_con_dato_region_anio = (
    wdi_long_region.dropna(subset=["Value"])
    .groupby(["Year", "Region"])["Country Code"]
    .nunique()
    .unstack("Region")
)

pct_region_anio = paises_con_dato_region_anio.div(n_paises_por_region, axis=1)
pct_region_anio.columns = [f"pct_{col}" for col in pct_region_anio.columns]

disponibilidad_anio = disponibilidad_anio.join(pct_region_anio).fillna(0)
# fillna(0) aquí es correcto: son porcentajes, no etiquetas, y 0% es un valor real
# (esa región no reportó nada ese año), no un "sin_dato"

disponibilidad_anio.shape

(66, 9)

In [71]:
disponibilidad_anio.head(80)

,n_paises_con_dato,pct_paises_con_dato,pct_East Asia & Pacific,pct_Europe & Central Asia,pct_Latin America & Caribbean,pct_Middle East & North Africa,pct_North America,pct_South Asia,pct_Sub-Saharan Africa
1960,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1961,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1962,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1963,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1964,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...
2021,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2022,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2023,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
2024,217,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0


## Graficamos distribución de los porcentajes de cobertura 


In [72]:
df_plot = disponibilidad_indicador_full.reset_index().rename(columns={"index": "Código WDI"})

fig_cobertura_indicador = px.histogram(
    df_plot,
    x="pct_paises_con_dato",
    nbins=50,
    hover_data=["Código WDI", "n_paises_con_dato", "primer_anio", "ultimo_anio"],
    labels={"pct_paises_con_dato": "% de países soberanos con al menos un dato"},
)

fig_cobertura_indicador.update_layout(
    title="Cobertura por indicador: ¿a cuántos países alcanza cada variable?",
    xaxis_title="% de países soberanos con al menos un dato",
    yaxis_title="Cantidad de indicadores",
    bargap=0.05,
    margin=dict(b=110),  # más espacio abajo para que entre la nota
)

fig_cobertura_indicador.add_annotation(
    text=(
        "Nota: cada barra agrupa indicadores según el % de los 217 países soberanos<br>"
        "que reportan al menos un dato en algún año (1960-2025). n=974 indicadores candidatos."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.30,
    showarrow=False,
    font=dict(size=11, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_indicador", fig_cobertura_indicador))
fig_cobertura_indicador.show()

Cada barra agrupa indicadores según su % de cobertura entre países. La distribución es aproximadamente bimodal: un grupo grande se concentra en coberturas altas (~0.8–1.0), hay un grupo secundario menor en coberturas medias-bajas (~0.3–0.55), y un valle relativo alrededor de 0.55–0.65 con relativamente pocos indicadores. Ese valle es un primer candidato visual para ubicar el umbral de corte, aunque conviene confirmarlo con los percentiles exactos (`disponibilidad_indicador_full["pct_paises_con_dato"].describe()` o `.quantile(...)`) antes de fijarlo definitivamente.

In [73]:
percentiles_finos = disponibilidad_indicador_full["pct_paises_con_dato"].quantile(
    [0.05, 0.10, 0.25, 0.40, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.90, 0.95]
)

resumen_cobertura_indicador = pd.concat([
    disponibilidad_indicador_full["pct_paises_con_dato"].describe(),
    percentiles_finos.rename("percentiles_finos"),
])

resumen_cobertura_indicador

count    980.000000
mean       0.805478
std        0.184025
min        0.004608
25%        0.737327
50%        0.861751
75%        0.935484
max        1.000000
0.05       0.419355
0.1        0.511060
0.25       0.737327
0.4        0.834101
0.5        0.861751
0.55       0.870968
0.6        0.889401
0.65       0.912442
0.7        0.921659
0.75       0.935484
0.9        0.986175
0.95       1.000000
dtype: float64

In [74]:
umbrales_candidatos = np.arange(0, 1.01, 0.01)

pct_retenido = [
    (disponibilidad_indicador_full["pct_paises_con_dato"] >= t).mean()
    for t in umbrales_candidatos
]

df_curva_umbral = pd.DataFrame({
    "umbral": umbrales_candidatos,
    "pct_indicadores_retenidos": pct_retenido,
})

fig_curva_umbral_indicador = px.line(
    df_curva_umbral,
    x="umbral",
    y="pct_indicadores_retenidos",
    labels={
        "umbral": "Umbral de corte candidato (pct_paises_con_dato mínimo exigido)",
        "pct_indicadores_retenidos": "% de indicadores retenidos",
    },
)

fig_curva_umbral_indicador.add_vrect(
    x0=0.55, x1=0.65,
    fillcolor="gray", opacity=0.2, line_width=0,
    annotation_text="valle", annotation_position="top left",
)

fig_curva_umbral_indicador.update_layout(
    title="Sensibilidad del universo de indicadores al umbral de corte de cobertura",
    yaxis_tickformat=".0%",
    margin=dict(b=80),
)

figuras_reporte.append(("curva_umbral_indicador", fig_curva_umbral_indicador))
fig_curva_umbral_indicador.show()

Acá lo que quisimos fue imitar una lógica análoga a la curva "ROC" 
Eje X: el umbral de exigencia que le pondríamos a un indicador para conservarlo ("solo me quedo con indicadores que cubran al menos X% de los países").
Eje Y: qué % de los 980 indicadores sobrevive con esa exigencia. El porcentaje es qué proporción de los 217 países  tienen al menos un dato de ese indicador en algún año

Lo que concluimos es que se exige que un indicador tenga dato en al menos el 60% de los 217 países (en algún momento entre 1960 y 2025), nos quedamos con el 84% de los 980 indicadores candidatos. SIN EMBARGO necesitamos más información sobre cómo está distribuida la disponibilidad de datos en el tiempo y en los países

## Gráfico: cobertura por país
Acá buscaremos grafica pct_indicadores_con_dato por país, ordenado y coloreado por región.

Resultado esperado: un gráfico de barras horizontales, uno por país, ordenado de menor a mayor cobertura, coloreado por Region, con hover mostrando el código de país y sus años de cobertura.

In [75]:
df_pais_plot = disponibilidad_pais.reset_index().rename(columns={"index": "Country Code"})
df_pais_plot = df_pais_plot.sort_values("pct_indicadores_con_dato")

fig_cobertura_pais = px.bar(
    df_pais_plot,
    x="pct_indicadores_con_dato",
    y="Country Code",
    color="Region",
    orientation="h",
    hover_data=["n_indicadores_con_dato", "primer_anio", "ultimo_anio"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_cobertura_pais.update_layout(
    title="Cobertura por país: ¿qué proporción de indicadores reporta cada país?",
    xaxis_title="% de indicadores con al menos un dato",
    yaxis_title="País (código ISO3)",
    height=3200,
    margin=dict(b=110),
)

fig_cobertura_pais.add_annotation(
    text=(
        "Nota: cada barra representa uno de los 217 países soberanos. El % indica cuántos<br>"
        "de los 974 indicadores candidatos tienen al menos un dato para ese país (1960-2025)."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.03,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_pais", fig_cobertura_pais))
fig_cobertura_pais.show()

Los países con cobertura más baja no están distribuidos al azar: se concentran en microestados y territorios dependientes, no en países soberanos "grandes" con problemas de reporte. se aprecia facilmente en la colas bajas: Gibraltar (GIB), Isla de Man (IMN), Groenlandia (GRL), Islas Caimán (CYM), Islas Vírgenes (VIR), Turcas y Caicos (TCA), Aruba (ABW), Antigua y Barbuda (ATG), Granada (GRD) — todos con cobertura por debajo de ~0.4-0.5. 
Creo que acá voy a tener que redefinir las unidades de estudio y dejar países en sentido estricto, y no "unidades económicas".
Es por eso que más adelante voy a importar una base de mimebros ONU para hacer un filtro

## Gráfico: Cobertura por región
el gráfico anterior muestra cada país individualmente; este resume la dispersión dentro de cada región, que es lo que hace falta para detectar si el problema de cobertura es sistemático de una región

In [76]:
fig_dispersion_region = px.box(
    df_pais_plot,
    x="Region",
    y="pct_indicadores_con_dato",
    points="all",
    hover_data=["Country Code", "n_indicadores_con_dato"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_dispersion_region.update_layout(
    title="Dispersión de cobertura por región: ¿hay regiones sistemáticamente peor cubiertas?",
    xaxis_title="Región (Banco Mundial)",
    yaxis_title="% de indicadores con al menos un dato",
    margin=dict(b=210),
)

fig_dispersion_region.add_annotation(
    text=(
        "Nota: cada punto es un país soberano; la caja muestra la mediana y el rango<br>"
        "intercuartílico de cobertura de indicadores dentro de cada región del Banco Mundial."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.80,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_cobertura_region", fig_dispersion_region))
fig_dispersion_region.show()

Los boxplots presentan por región, la misma información que pudimos ver en el gráfico de cobertura por país. De hecho fue realizado utilizando el mismo dataframe, pero utilizando por referencia a las regiones. Este gráfico permite apreciar nuevamente que la región geográfica en sí no parece ser el criterio que mejor discrimina cobertura baja. 

In [77]:
df_pais_plot.columns

Index(['Country Code', 'n_indicadores_con_dato', 'primer_anio', 'ultimo_anio',
       'pct_indicadores_con_dato', 'Region'],
      dtype='str')

In [78]:
df_pais_plot.Region.unique()

<ArrowStringArray>
[ 'Latin America & Caribbean',      'Europe & Central Asia',
        'East Asia & Pacific',              'North America',
         'Sub-Saharan Africa', 'Middle East & North Africa',
                 'South Asia']
Length: 7, dtype: str

# Sección 2: Filtro de miembros ONU y regiones M49

## Unimos la codificación de países miembros de onu con M49 a nuestro df del BM (WDI)

In [79]:
df_regiones_onu = pd.read_excel("df_regiones_miembros_onu.xlsx")
df_regiones_onu.head()

,Member State,M49_country,ISO-alpha3,Other Names,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code,nombre_norm_principal,nombre_norm_alt,cow_code_countryVdem
0,United States,840,USA,"USA, U.S.A., United States of America",United States of America,19,Americas,21,Northern America,US,united states,usa u s a united states of america,2
1,Australia,36,AUS,Commonwealth of Australia,Australia,9,Oceania,53,Australia and New Zealand,AU,australia,commonwealth of australia,900
2,Djibouti,262,DJI,Republic of Djibouti,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ,djibouti,republic of djibouti,522
3,Ghana,288,GHA,Republic of Ghana,Ghana,2,Africa,202,Sub-Saharan Africa,GH,ghana,republic of ghana,452
4,Kiribati,296,KIR,Republic of Kiribati,Kiribati,9,Oceania,57,Micronesia,KI,kiribati,republic of kiribati,946


In [80]:
iso_onu = set(df_regiones_onu["ISO-alpha3"])

# Nueva lista: soberanos WDI que SÍ son Estados miembros de la ONU (no sobreescribe paises_soberanos)
paises_onu = [c for c in paises_soberanos if c in iso_onu]

# Auditoría 1: soberanos WDI que NO matchean con la ONU (territorios, no miembros, etc.)
wdi_sin_match_onu = sorted(set(paises_soberanos) - iso_onu)

# Auditoría 2: Estados miembros ONU que NO aparecen como soberanos en WDI (ausencia total de datos)
onu_sin_match_wdi = df_regiones_onu.loc[~df_regiones_onu["ISO-alpha3"].isin(paises_soberanos)]

print(f"Países soberanos WDI (original): {len(paises_soberanos)}")
print(f"Países ONU tras el match: {len(paises_onu)}")
print(f"\nWDI sin match en ONU ({len(wdi_sin_match_onu)}):")
print(wdi_sin_match_onu)

print(f"\nEstados ONU sin match en WDI ({len(onu_sin_match_wdi)}):")
onu_sin_match_wdi[["Member State", "ISO-alpha3"]]

Países soberanos WDI (original): 217
Países ONU tras el match: 193

WDI sin match en ONU (24):
['ABW', 'ASM', 'BMU', 'CHI', 'CUW', 'CYM', 'FRO', 'GIB', 'GRL', 'GUM', 'HKG', 'IMN', 'MAC', 'MAF', 'MNP', 'NCL', 'PRI', 'PSE', 'PYF', 'SXM', 'TCA', 'VGB', 'VIR', 'XKX']

Estados ONU sin match en WDI (0):


,Member State,ISO-alpha3


## Recalculamos

In [81]:
n_paises_total_onu = len(paises_onu)
print(f"Países totales (ONU): {n_paises_total_onu}  |  Países totales (original): {n_paises_total}")

Países totales (ONU): 193  |  Países totales (original): 217


In [82]:
wdi_long_onu = wdi_long[wdi_long["Country Code"].isin(paises_onu)]

paises_con_dato_full_onu = (
    wdi_long_onu.dropna(subset=["Value"])
    .groupby("Indicator Code")["Country Code"]
    .nunique()
    .rename("n_paises_con_dato_onu")
)

disponibilidad_indicador_full_onu = (
    pd.DataFrame(index=indicadores_candidatos)
    .join(paises_con_dato_full_onu)
)
disponibilidad_indicador_full_onu["n_paises_con_dato_onu"] = disponibilidad_indicador_full_onu["n_paises_con_dato_onu"].fillna(0)
disponibilidad_indicador_full_onu["pct_paises_con_dato_onu"] = (
    disponibilidad_indicador_full_onu["n_paises_con_dato_onu"] / n_paises_total_onu
)

## Cobertura por país con universo ONU

In [83]:
disponibilidad_pais_onu = disponibilidad_pais.loc[paises_onu].copy()

# Anexamos columnas M49 (descriptivas, igual que se hizo con "Region" en el bloque 21)
m49_por_pais = df_regiones_onu.set_index("ISO-alpha3")[
    ["M49_region", "Region Name", "M49_subregion", "Sub-region Name"]
]
disponibilidad_pais_onu = disponibilidad_pais_onu.join(m49_por_pais)

disponibilidad_pais_onu.shape

(193, 9)

In [84]:
disponibilidad_pais_onu.head()

,n_indicadores_con_dato,primer_anio,ultimo_anio,pct_indicadores_con_dato,Region,M49_region,Region Name,M49_subregion,Sub-region Name
AFG,876,1960,2025,0.893878,Middle East & North Africa,142,Asia,34,Southern Asia
AGO,940,1960,2025,0.959184,Sub-Saharan Africa,2,Africa,202,Sub-Saharan Africa
ALB,943,1960,2025,0.962245,Europe & Central Asia,150,Europe,39,Southern Europe
AND,429,1960,2025,0.437755,Europe & Central Asia,150,Europe,39,Southern Europe
ARE,807,1960,2025,0.823469,Middle East & North Africa,142,Asia,145,Western Asia


In [85]:
df_pais_plot_onu = disponibilidad_pais_onu.reset_index().rename(columns={"index": "Country Code"})
df_pais_plot_onu = df_pais_plot_onu.sort_values("pct_indicadores_con_dato")

fig_cobertura_pais_onu = px.bar(
    df_pais_plot_onu,
    x="pct_indicadores_con_dato",
    y="Country Code",
    color="Region Name",
    orientation="h",
    hover_data=["n_indicadores_con_dato", "primer_anio", "ultimo_anio", "Sub-region Name"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_cobertura_pais_onu.update_layout(
    title="Cobertura por país (universo ONU, M49): ¿qué proporción de indicadores reporta cada país?",
    xaxis_title="% de indicadores con al menos un dato",
    yaxis_title="País (código ISO3)",
    height=3200,
    margin=dict(b=110),
)

fig_cobertura_pais_onu.add_annotation(
    text=(
        "Nota: cada barra representa uno de los 193 Estados miembros de la ONU. El % indica<br>"
        "cuántos de los 974 indicadores candidatos tienen al menos un dato para ese país (1960-2025)."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.03,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_pais_onu", fig_cobertura_pais_onu))
fig_cobertura_pais_onu.show()

## Distribución de porcentajes de cobertura para miembros ONU

In [86]:
fig_cobertura_indicador_comparada = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("WDI (217 países)", "ONU (193 países)"),
    vertical_spacing=0.12,
)

fig_cobertura_indicador_comparada.add_trace(
    go.Histogram(
        x=disponibilidad_indicador_full["pct_paises_con_dato"],
        nbinsx=50,
        name="WDI",
    ),
    row=1, col=1,
)
fig_cobertura_indicador_comparada.add_trace(
    go.Histogram(
        x=disponibilidad_indicador_full_onu["pct_paises_con_dato_onu"],
        nbinsx=50,
        name="ONU",
    ),
    row=2, col=1,
)

fig_cobertura_indicador_comparada.update_yaxes(title_text="Cantidad de indicadores", row=1, col=1)
fig_cobertura_indicador_comparada.update_yaxes(title_text="Cantidad de indicadores", row=2, col=1)
fig_cobertura_indicador_comparada.update_xaxes(title_text="% de países con al menos un dato", row=2, col=1)

fig_cobertura_indicador_comparada.update_layout(
    title="Cobertura por indicador: universo WDI vs. universo ONU (n=974 indicadores candidatos)",
    bargap=0.05,
    height=700,
    margin=dict(b=130),
    showlegend=False,
)

fig_cobertura_indicador_comparada.add_annotation(
    text=(
        "Nota: cada barra agrupa indicadores según el % de países que reportan al menos un dato<br>"
        "en algún año (1960-2025). Panel superior: 217 países soberanos. Panel inferior: 193 Estados miembros de la ONU."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.18,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left",
)

figuras_reporte.append(("cobertura_por_indicador_comparada_wdi_onu", fig_cobertura_indicador_comparada))
fig_cobertura_indicador_comparada.show()

En el eje X está el % de países que reportan cada indicador; en el eje Y, cuántos de los 974 indicadores caen en cada franja de cobertura. El panel de arriba es el universo WDI completo (217 países); el de abajo, solo miembros ONU (193).

Mejora muchísimo la cobertura.
el panel ONU (abajo) está mucho más "cargado hacia la derecha" que el WDI completo (arriba). En la barra del extremo derecho (cobertura 100%): en WDI hay 88 indicadores ahí; en ONU hay 267

In [87]:
umbrales_candidatos = np.arange(0, 1.01, 0.01)

pct_retenido_wdi = [
    (disponibilidad_indicador_full["pct_paises_con_dato"] >= t).mean()
    for t in umbrales_candidatos
]
pct_retenido_onu = [
    (disponibilidad_indicador_full_onu["pct_paises_con_dato_onu"] >= t).mean()
    for t in umbrales_candidatos
]

fig_curva_umbral_comparada = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("WDI (217 países)", "ONU (193 países)"),
    vertical_spacing=0.12,
)

fig_curva_umbral_comparada.add_trace(
    go.Scatter(x=umbrales_candidatos, y=pct_retenido_wdi, mode="lines", name="WDI"),
    row=1, col=1,
)
fig_curva_umbral_comparada.add_trace(
    go.Scatter(x=umbrales_candidatos, y=pct_retenido_onu, mode="lines", name="ONU"),
    row=2, col=1,
)

for row in [1, 2]:
    fig_curva_umbral_comparada.add_vrect(
        x0=0.55, x1=0.65,
        fillcolor="gray", opacity=0.2, line_width=0,
        row=row, col=1,
    )

fig_curva_umbral_comparada.update_yaxes(title_text="% de indicadores retenidos", tickformat=".0%", row=1, col=1)
fig_curva_umbral_comparada.update_yaxes(title_text="% de indicadores retenidos", tickformat=".0%", row=2, col=1)
fig_curva_umbral_comparada.update_xaxes(title_text="Umbral de corte candidato (% de países con al menos un dato)", row=2, col=1)

fig_curva_umbral_comparada.update_layout(
    title="Sensibilidad al umbral de corte de cobertura: universo WDI vs. universo ONU",
    height=700,
    margin=dict(b=110),
    showlegend=False,
)

figuras_reporte.append(("curva_umbral_indicador_comparada_wdi_onu", fig_curva_umbral_comparada))
fig_curva_umbral_comparada.show()

El umbral de corte que definamos (probablemente 0.55-0.65) es estable ante la elección de universo

## Dispersión de cobertura por región y subregión (M49)

In [88]:
fig_dispersion_region_onu = px.box(
    df_pais_plot_onu,
    x="Region Name",
    y="pct_indicadores_con_dato",
    points="all",
    hover_data=["Country Code", "n_indicadores_con_dato", "Sub-region Name"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_dispersion_region_onu.update_layout(
    title="Dispersión de cobertura por región M49 (Region Name): ¿hay regiones sistemáticamente peor cubiertas?",
    xaxis_title="Región (M49 — continental)",
    yaxis_title="% de indicadores con al menos un dato",
    margin=dict(b=150),
)

fig_dispersion_region_onu.add_annotation(
    text=(
        "Nota: cada punto es un Estado miembro de la ONU (193); la caja muestra mediana y rango<br>"
        "intercuartílico de cobertura de indicadores dentro de cada región continental M49."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.35,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_cobertura_region_m49", fig_dispersion_region_onu))
fig_dispersion_region_onu.show()

En línea con lo anterior, aquí podemos apreciar que la caja con datos extraídos de oceanía para la base de datos del WDI (Q1 a Q3) se ubica por debajo de los restantes continentes. Sin embargo, el 50% de sus países tienen una cobertura entre el 66 y 88 %. Los demás continentes están más concentrados, y todos tienen sus rangos intercuartílicos sobre 79%.


In [89]:
fig_dispersion_subregion_onu = px.box(
    df_pais_plot_onu,
    x="Sub-region Name",
    y="pct_indicadores_con_dato",
    points="all",
    hover_data=["Country Code", "n_indicadores_con_dato", "Region Name"],
    labels={"pct_indicadores_con_dato": "% de indicadores con al menos un dato"},
)

fig_dispersion_subregion_onu.update_layout(
    title="Dispersión de cobertura por subregión M49 (Sub-region Name)",
    xaxis_title="Subregión (M49)",
    yaxis_title="% de indicadores con al menos un dato",
    margin=dict(b=220),
)
fig_dispersion_subregion_onu.update_xaxes(tickangle=45)

fig_dispersion_subregion_onu.add_annotation(
    text=(
        "Nota: cada punto es un Estado miembro de la ONU (193); algunas subregiones tienen muy<br>"
        "pocos países (ej. Micronesia, Polynesia) — leer su dispersión con cautela por bajo n."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.70,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_cobertura_subregion_m49", fig_dispersion_subregion_onu))
fig_dispersion_subregion_onu.show()

## Documentación del match WDI ↔ ONU

**Objetivo:** dejar documentada la comparación bidireccional entre el universo WDI (217 unidades soberanas) y el universo ONU (193 Estados miembros), ya calculada más arriba (`wdi_sin_match_onu`, `onu_sin_match_wdi`).

La exportación de estas tablas a `disponibilidad_full.xlsx` (hojas `match_wdi_no_onu` y `match_onu_sin_wdi`) se hace en el bloque de exportación, al final del notebook.

In [90]:
# Dirección 1: códigos WDI (soberanos) que NO son miembros ONU
nombre_por_codigo = country_meta.set_index("Country Code")["Short Name"]  # ajustar nombre de columna si difiere

df_wdi_sin_match_onu = pd.DataFrame({
    "Country Code": wdi_sin_match_onu,
})
df_wdi_sin_match_onu["Short Name"] = df_wdi_sin_match_onu["Country Code"].map(nombre_por_codigo)

# Dirección 2: miembros ONU sin correspondencia soberana en WDI (ya calculado en la sección ONU)
df_onu_sin_match_wdi = onu_sin_match_wdi[["Member State", "ISO-alpha3"]].reset_index(drop=True)

print(f"[Dirección 1] Códigos WDI sin match en ONU ({len(df_wdi_sin_match_onu)}):")
print(df_wdi_sin_match_onu.to_string(index=False))
print()


[Dirección 1] Códigos WDI sin match en ONU (24):
Country Code                Short Name
         ABW                     Aruba
         ASM            American Samoa
         BMU                   Bermuda
         CHI           Channel Islands
         CUW                   Curaçao
         CYM            Cayman Islands
         FRO             Faroe Islands
         GIB                 Gibraltar
         GRL                 Greenland
         GUM                      Guam
         HKG      Hong Kong SAR, China
         IMN               Isle of Man
         MAC          Macao SAR, China
         MAF  St. Martin (French part)
         MNP  Northern Mariana Islands
         NCL             New Caledonia
         PRI          Puerto Rico (US)
         PSE        West Bank and Gaza
         PYF          French Polynesia
         SXM Sint Maarten (Dutch part)
         TCA  Turks and Caicos Islands
         VGB    British Virgin Islands
         VIR       Virgin Islands (US)
         XKX   

In [91]:
print(f"[Dirección 2] Miembros ONU sin match en WDI ({len(df_onu_sin_match_wdi)}):")
print(df_onu_sin_match_wdi.to_string(index=False))


[Dirección 2] Miembros ONU sin match en WDI (0):
Empty DataFrame
Columns: [Member State, ISO-alpha3]
Index: []


# Sección 3: Análisis de recorte temporal

## Gráfico: curva de cobertura agregada por año

In [92]:
n_anios_totales = wdi_long["Year"].nunique()
celdas_posibles_total = n_paises_total * n_indicadores_total

celdas_con_dato_anio = wdi_long.dropna(subset=["Value"]).groupby("Year").size()

df_anio_plot = disponibilidad_anio.reset_index().rename(columns={"index": "Year"})
df_anio_plot["densidad_datos"] = (
    df_anio_plot["Year"].map(celdas_con_dato_anio).fillna(0) / celdas_posibles_total
)

fig_densidad_anio = px.line(
    df_anio_plot,
    x="Year",
    y="densidad_datos",
    markers=True,
    labels={"densidad_datos": "% de celdas país-indicador con dato"},
)

fig_densidad_anio.update_layout(
    title="Sensibilidad de la cobertura: ¿qué proporción de la matriz país-indicador está completa?",
    xaxis_title="Año",
    yaxis_title="% de celdas país-indicador con dato",
    yaxis_range=[0, 1.05],
    margin=dict(b=130),
)

fig_densidad_anio.add_annotation(
    text=(
        "Nota: Este gráfico busca medir para cada año qué % de las 217×974 celdas<br>"
        "posibles país-indicador tienen realmente un valor. Es una medida de densidad, no de presencia binaria."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.42,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_panel_por_anio", fig_densidad_anio))
fig_densidad_anio.show()

Acá podemos apreciar que en los años 90 la cobertura pega un salto, y que desde el 2000 en adelante esta curva empieza a estabilizarse. Esto nos proporciona información para establecer un año de corte prudente

## Heatmap con cobertura regional en el tiempo — universo ONU / M49

In [93]:
wdi_long_region_onu = wdi_long_onu.merge(
    df_regiones_onu[["ISO-alpha3", "Region Name"]],
    left_on="Country Code", right_on="ISO-alpha3", how="left"
)

n_paises_por_region_onu = df_regiones_onu.groupby("Region Name")["ISO-alpha3"].nunique()
celdas_posibles_region_onu = n_paises_por_region_onu * n_indicadores_total

celdas_con_dato_region_anio_onu = (
    wdi_long_region_onu.dropna(subset=["Value"])
    .groupby(["Year", "Region Name"])
    .size()
    .unstack("Region Name")
)

densidad_region_anio_onu = celdas_con_dato_region_anio_onu.div(celdas_posibles_region_onu, axis=1).fillna(0)
matriz_region_anio_onu = densidad_region_anio_onu.T

fig_heatmap_region_onu = px.imshow(
    matriz_region_anio_onu,
    labels=dict(x="Año", y="Región (M49)", color="% celdas con dato"),
    x=matriz_region_anio_onu.columns,
    y=matriz_region_anio_onu.index,
    color_continuous_scale="RdYlGn",
    zmin=0,
    zmax=float(matriz_region_anio_onu.values.max()),
    aspect="auto",
)

fig_heatmap_region_onu.update_layout(
    title="Densidad regional M49 en el tiempo (Region Name): ¿dónde y cuándo se concentran los huecos?",
    margin=dict(b=130),
)

fig_heatmap_region_onu.add_annotation(
    text=(
        "Nota: cada celda muestra, para esa región M49 y ese año, qué % de las celdas país-<br>"
        "indicador posibles (países ONU de la región × 974 indicadores) tienen realmente un dato."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.45,
    showarrow=False,
    font=dict(size=13, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_regional_heatmap_m49", fig_heatmap_region_onu))
fig_heatmap_region_onu.show()

In [94]:
fig_lineas_region_anio_onu = px.line(
    densidad_region_anio_onu.reset_index(),
    x="Year",
    y=densidad_region_anio_onu.columns.tolist(),
    labels={"Year": "Año", "value": "% de celdas país-indicador con dato", "variable": "Región"},
)
fig_lineas_region_anio_onu.update_layout(
    title="Disponibilidad regional M49 a lo largo del tiempo (líneas) — universo ONU",
    yaxis_tickformat=".0%",
    legend_title_text="Región",
    margin=dict(b=90),
)
fig_lineas_region_anio_onu.add_annotation(
    text="Nota: mismos datos que el heatmap de arriba — % de celdas país-indicador con dato dentro de cada región M49, para cada año.",
    xref="paper", yref="paper", x=0, y=-0.2, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("lineas_region_anio_onu", fig_lineas_region_anio_onu))
fig_lineas_region_anio_onu.show()

Este gráfico nos permite ver que desde los años 90 o 95 en adelante obtenemos mejor cobertura. Sin embargo, también nos permite ver que para los territorios de Oceanía, esta cobertura inicio con mayor retraso en el tiempo. Para el año 1995, por ejemplo, mientras que para este continente la cobertura es del 33%, para los restantes es superior al 45%. Tenemos que ampliar este aspecto para decidir qué hacer

In [95]:
def calcular_densidad_por_pais(datos_wdi, paises, indicadores, anio_inicio, anio_fin):
    sub = datos_wdi[
        datos_wdi["Country Code"].isin(paises) &
        datos_wdi["Indicator Code"].isin(indicadores) &
        (datos_wdi["Year"] >= anio_inicio) &
        (datos_wdi["Year"] <= anio_fin)
    ]
    n_celdas_por_pais = len(indicadores) * (anio_fin - anio_inicio + 1)
    celdas_con_dato_por_pais = (
        sub.dropna(subset=["Value"])
        .groupby("Country Code")
        .size()
    )
    densidad_pais = (
        pd.Series(index=paises, dtype=float)
        .fillna(0)
        .add(celdas_con_dato_por_pais.reindex(paises).fillna(0), fill_value=0)
        / n_celdas_por_pais
    )
    return densidad_pais

anio_min_disponible_oceania = int(wdi_long["Year"].min())
anio_max_disponible_oceania = int(wdi_long["Year"].max())

paises_oceania = df_pais_plot_onu.loc[df_pais_plot_onu["Region Name"] == "Oceania", "Country Code"].tolist()

densidad_pais_oceania = calcular_densidad_por_pais(
    wdi_long, paises_oceania, indicadores_candidatos, anio_min_disponible_oceania, anio_max_disponible_oceania
)

nombre_por_codigo_oceania = country_meta.set_index("Country Code")["Short Name"]

df_densidad_oceania = (
    densidad_pais_oceania
    .rename("densidad")
    .reset_index()
    .rename(columns={"index": "Country Code"})
)
df_densidad_oceania["Nombre"] = df_densidad_oceania["Country Code"].map(nombre_por_codigo_oceania)
df_densidad_oceania = df_densidad_oceania.sort_values("densidad", ascending=False)

fig_densidad_pais_oceania = px.bar(
    df_densidad_oceania,
    x="Nombre",
    y="densidad",
    labels={"densidad": "Proporción de datos cargados", "Nombre": "País"},
)

fig_densidad_pais_oceania.update_layout(
    title="Proporción de datos cargados por país — Oceanía (1960 - 2025)",
    xaxis_title="País",
    yaxis_title="Proporción de datos cargados",
    yaxis_tickformat=".0%",
    margin=dict(b=100),
)

fig_densidad_pais_oceania.add_annotation(
    text=(
        "Nota: de todos los datos que un país podría tener (cada indicador, en cada año), qué<br>"
        "proporción tiene efectivamente cargada."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.45,
    showarrow=False,
    font=dict(size=12, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_pais_oceania", fig_densidad_pais_oceania))
fig_densidad_pais_oceania.show()

Vemos dos grupos, los que tienen cobertura mayor a 40%, que son Estados con economías independientes , y luego países con economías pequeñas y que con autonomía monetaria limitada. Islas Marshall, Palau y Micronesia no tienen banco central propio, por ejemplo.

## Gráfico: dispersión de cobertura por década
boxplot de la densidad real de datos (% de celdas país-indicador con dato) agrupando los años en décadas.

resume la variabilidad año a año dentro de cada década, útil para ver si hay décadas con mayor inestabilidad de cobertura (no solo el promedio, sino la dispersión).

Resultado esperado: una caja por década, de 1960s a 2020s.

In [96]:
df_anio_plot["decada"] = (df_anio_plot["Year"] // 10 * 10).astype(str) + "s"
orden_decadas = sorted(df_anio_plot["decada"].unique())

fig_dispersion_decada = px.box(
    df_anio_plot,
    x="decada",
    y="densidad_datos",
    points="all",
    category_orders={"decada": orden_decadas},
    hover_data=["Year"],
    labels={"densidad_datos": "% de celdas país-indicador con dato"},
)

fig_dispersion_decada.update_layout(
    title="Estabilidad de la densidad de datos por década",
    xaxis_title="Década",
    yaxis_title="% de celdas país-indicador con dato",
    margin=dict(b=110),
)

fig_dispersion_decada.add_annotation(
    text=(
        "Nota: cada punto es un año dentro de la década; la caja muestra la dispersión<br>"
        "de la densidad real de datos (celdas país-indicador completas) entre los años de esa década."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.30,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("dispersion_densidad_decada", fig_dispersion_decada))
fig_dispersion_decada.show()

Acá podemos comparar el hancho de las cajas en cada década. Décadas con cajas anchas indican que la densidad de datos varió mucho año a año dentro de ese período. Sin embargo creemos que no es tan seguro tomar esta visualización como referencia, porque como vimos, hay muy pocos datos en los años anteriors a 1990. Entonces esta situación más homogenea, es homogenea en el sentido de que hay más datos faltantes. A la vez, en 2020 en adelante por supuesto habrá más variación, porque en las bases internacionales en general hay pocos datos para los años cercanos en el tiempo

## Gráfico: densidad real de datos, por indicador y por país

A diferencia del criterio binario usado antes ("¿tiene o no tiene al menos un dato?", que da 100% tanto para indicadores como para países, porque todos tienen al menos un valor en algún año), acá se mide la **densidad real**: qué proporción de las celdas país-año (para cada indicador) o indicador-año (para cada país) están efectivamente completas.

Objetivo: ver la distribución de densidad real —no solo la presencia binaria— para dimensionar cuán completo está realmente el panel más allá de que "algo" de dato exista.

Resultado esperado: dos distribuciones (por indicador y por país) del % de celdas con dato, más la densidad global del panel completo.

In [97]:
n_anios_totales = wdi_long["Year"].nunique()

celdas_indicador = wdi_long.dropna(subset=["Value"]).groupby("Indicator Code").size()
densidad_indicador = (
    celdas_indicador.reindex(indicadores_candidatos).fillna(0) / (n_paises_total * n_anios_totales)
)

celdas_pais = wdi_long.dropna(subset=["Value"]).groupby("Country Code").size()
densidad_pais = (
    celdas_pais.reindex(paises_soberanos).fillna(0) / (n_indicadores_total * n_anios_totales)
)

densidad_total = wdi_long["Value"].notna().sum() / len(wdi_long)

df_densidad = pd.concat([
    pd.DataFrame({"Código": densidad_indicador.index, "Densidad": densidad_indicador.values, "Nivel": "Por indicador"}),
    pd.DataFrame({"Código": densidad_pais.index, "Densidad": densidad_pais.values, "Nivel": "Por país"}),
])

fig_densidad_real = px.histogram(
    df_densidad,
    x="Densidad",
    facet_col="Nivel",
    nbins=30,
    labels={"Densidad": "% de celdas con dato"},
)

fig_densidad_real.update_layout(
    title=f"Densidad real del panel: proporción de celdas con dato (densidad global = {densidad_total:.1%})",
    margin=dict(b=130),
)

fig_densidad_real.add_annotation(
    text=(
        "Nota: 'Por indicador' = para cada indicador, % de las celdas país-año (217×"
        f"{n_anios_totales}) con dato. 'Por país' = para cada país, % de las celdas indicador-<br>"
        "año (974×" + str(n_anios_totales) + ") con dato. No es lo mismo que 'tener al menos un dato': mide cuán<br>"
        "completa está realmente la serie, no solo su existencia."
    ),
    xref="paper", yref="paper",
    x=0, y=-0.45,
    showarrow=False,
    font=dict(size=10, color="gray"),
    align="left",
)

figuras_reporte.append(("densidad_real_indicador_pais", fig_densidad_real))
fig_densidad_real.show()

La densidad global resume qué proporción de toda la matriz país-indicador-año está efectivamente completa. Las dos distribuciones muestran si esa densidad es pareja (histogramas concentrados) o si hay indicadores/países con densidad muy baja aunque figuren como "con dato" en los gráficos anteriores. 

Panel izquierdo (por indicador): hay un pico grande en 1.0 — un grupo de 79 indicadores que tienen dato en prácticamente todas las celdas país-año posibles (probablemente los "clásicos": población, PBI, esperanza de vida).
Panel derecho (por país): acá no hay ningún país que se acerque a 1.0. El máximo que alcanza cualquier país está apenas por encima de 0.55. Que ningún país se acerque al 100% creo que debe tener que ver con la línea del tiempo, que es  lo que vamos a evaluar a continuación



## Curva de cobertura agregada por año: comparación universo ONU vs. WDI

In [98]:
def calcular_densidad_por_anio(datos_wdi, paises, indicadores):
    sub = datos_wdi[
        datos_wdi["Country Code"].isin(paises) &
        datos_wdi["Indicator Code"].isin(indicadores)
    ]
    n_celdas_por_anio = len(paises) * len(indicadores)
    celdas_con_dato_por_anio = (
        sub.dropna(subset=["Value"])
        .groupby("Year")
        .size()
    )
    anios_completos = range(int(datos_wdi["Year"].min()), int(datos_wdi["Year"].max()) + 1)
    densidad_anio = (
        celdas_con_dato_por_anio.reindex(anios_completos).fillna(0) / n_celdas_por_anio
    )
    return densidad_anio

densidad_anio_onu = calcular_densidad_por_anio(wdi_long, paises_onu, indicadores_candidatos)
densidad_anio_wdi = calcular_densidad_por_anio(wdi_long, paises_soberanos, indicadores_candidatos)

df_densidad_anio_comparada = pd.concat([
    pd.DataFrame({"Year": densidad_anio_onu.index, "densidad": densidad_anio_onu.values, "Universo": "ONU (193 países)"}),
    pd.DataFrame({"Year": densidad_anio_wdi.index, "densidad": densidad_anio_wdi.values, "Universo": "WDI (217 países)"}),
])

df_densidad_anio_comparada.head()

,Year,densidad,Universo
0,1960,0.129433,ONU (193 países)
1,1961,0.138675,ONU (193 países)
2,1962,0.143412,ONU (193 países)
3,1963,0.145538,ONU (193 países)
4,1964,0.147193,ONU (193 países)


In [99]:
fig_densidad_anio_comparada = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Universo ONU (193 países)", "Universo WDI (217 países)"),
    vertical_spacing=0.12,
)

df_onu_anio = df_densidad_anio_comparada[df_densidad_anio_comparada["Universo"] == "ONU (193 países)"]
df_wdi_anio = df_densidad_anio_comparada[df_densidad_anio_comparada["Universo"] == "WDI (217 países)"]

fig_densidad_anio_comparada.add_trace(
    go.Scatter(x=df_onu_anio["Year"], y=df_onu_anio["densidad"], mode="lines", name="ONU"),
    row=1, col=1,
)
fig_densidad_anio_comparada.add_trace(
    go.Scatter(x=df_wdi_anio["Year"], y=df_wdi_anio["densidad"], mode="lines", name="WDI"),
    row=2, col=1,
)

for row in [1, 2]:
    fig_densidad_anio_comparada.add_vline(x=1990, line_dash="dash", line_color="gray", row=row, col=1)

fig_densidad_anio_comparada.update_yaxes(title_text="% celdas con dato", tickformat=".0%", row=1, col=1)
fig_densidad_anio_comparada.update_yaxes(title_text="% celdas con dato", tickformat=".0%", row=2, col=1)
fig_densidad_anio_comparada.update_xaxes(title_text="Año", row=2, col=1)

fig_densidad_anio_comparada.update_layout(
    title="Densidad del panel por año: universo ONU vs. universo WDI",
    height=700,
    showlegend=False,
)

figuras_reporte.append(("densidad_anio_comparada_onu_wdi", fig_densidad_anio_comparada))
fig_densidad_anio_comparada.show()

A diferencia de la curva de sensibilidad por año de inicio (más abajo), acá cada punto es la densidad de un único año puntual, no de un rango acumulado desde ese año. 
vemos que desde los años 90 se mejroa la recolección de datos (o al menos hay más presencia de ellos) y que esto disminuye sobre el final (esperable por el tiempo que tardan los países en reportar sus estadísticas).

Nos quedaremos con este gráfico que se presenta a continuación sin la comparación:

In [100]:
df_densidad_anio_onu_solo = pd.DataFrame({
    "Year": densidad_anio_onu.index,
    "densidad_real": densidad_anio_onu.values,
})

fig_densidad_anio_onu = px.line(
    df_densidad_anio_onu_solo,
    x="Year",
    y="densidad_real",
    markers=True,
    labels={"densidad_real": "% de celdas país-indicador con dato"},
)
fig_densidad_anio_onu.update_layout(
    title=dict(text="% de cobertura de datos a lo largo del tiempo (WDI)", x=0.5, xanchor="center"),
    xaxis_title="Año",
    yaxis_title="% de celdas país-indicador con dato",
    yaxis_range=[0, 1],
    margin=dict(b=90),
)
fig_densidad_anio_onu.add_annotation(
    text="Nota: % de la matriz países×indicadores que está completa en cada año. Se considera únicamente miembros ONU de la base WDI.",
    xref="paper", yref="paper", x=0, y=-0.22, showarrow=False,
    font=dict(size=12, color="gray"), align="left",
)

figuras_reporte.append(("densidad_anio_onu_solo", fig_densidad_anio_onu))
fig_densidad_anio_onu.show()

## Curva de sensibilidad de la densidad del panel al año de inicio

In [101]:
anio_max_disponible = int(wdi_long["Year"].max())
anio_min_disponible = int(wdi_long["Year"].min())

anios_inicio_candidatos = list(range(
    (anio_min_disponible // 5) * 5, anio_max_disponible + 1, 5
))

def calcular_densidad_panel(datos_wdi, paises, indicadores, anio_inicio, anio_fin):
    sub = datos_wdi[
        (datos_wdi["Year"] >= anio_inicio) &
        (datos_wdi["Year"] <= anio_fin) &
        (datos_wdi["Country Code"].isin(paises)) &
        (datos_wdi["Indicator Code"].isin(indicadores))
    ]
    n_celdas_totales = len(paises) * len(indicadores) * (anio_fin - anio_inicio + 1)
    n_celdas_con_dato = sub["Value"].notna().sum()
    return n_celdas_con_dato / n_celdas_totales

registros_curva = []
for anio_inicio in anios_inicio_candidatos:
    densidad_onu = calcular_densidad_panel(
        wdi_long, paises_onu, indicadores_candidatos, anio_inicio, anio_max_disponible
    )
    densidad_wdi = calcular_densidad_panel(
        wdi_long, paises_soberanos, indicadores_candidatos, anio_inicio, anio_max_disponible
    )
    registros_curva.append({"anio_inicio": anio_inicio, "densidad": densidad_onu, "Universo": "ONU (193 países)"})
    registros_curva.append({"anio_inicio": anio_inicio, "densidad": densidad_wdi, "Universo": "WDI (217 países)"})

df_curva_cobertura = pd.DataFrame(registros_curva)
df_curva_cobertura.head()

,anio_inicio,densidad,Universo
0,1960,0.419065,ONU (193 países)
1,1960,0.393370,WDI (217 países)
2,1965,0.441869,ONU (193 países)
3,1965,0.414506,WDI (217 países)
4,1970,0.467365,ONU (193 países)


In [102]:
fig_curva_cobertura = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=("Universo ONU (193 países)", "Universo WDI (217 países)"),
    vertical_spacing=0.12,
)

df_onu_curva = df_curva_cobertura[df_curva_cobertura["Universo"] == "ONU (193 países)"]
df_wdi_curva = df_curva_cobertura[df_curva_cobertura["Universo"] == "WDI (217 países)"]

fig_curva_cobertura.add_trace(
    go.Scatter(x=df_onu_curva["anio_inicio"], y=df_onu_curva["densidad"], mode="lines+markers", name="ONU"),
    row=1, col=1,
)
fig_curva_cobertura.add_trace(
    go.Scatter(x=df_wdi_curva["anio_inicio"], y=df_wdi_curva["densidad"], mode="lines+markers", name="WDI"),
    row=2, col=1,
)

for row in [1, 2]:
    fig_curva_cobertura.add_vline(x=1990, line_dash="dash", line_color="gray", row=row, col=1)

fig_curva_cobertura.update_yaxes(title_text="Densidad del panel", tickformat=".0%", row=1, col=1)
fig_curva_cobertura.update_yaxes(title_text="Densidad del panel", tickformat=".0%", row=2, col=1)
fig_curva_cobertura.update_xaxes(title_text="Año de inicio candidato", row=2, col=1)

fig_curva_cobertura.update_layout(
    title="Sensibilidad de la densidad del panel (1960-2025)",
    height=700,
    showlegend=False,
)

figuras_reporte.append(("curva_sensibilidad_anio_inicio", fig_curva_cobertura))
fig_curva_cobertura.show()

Cada punto de la curva responde a la pregunta: "si arranco mi panel de datos en el año X (en vez de otro año), ¿qué porcentaje de todas las celdas posibles (país × indicador × año, desde X hasta el último año disponible) tienen efectivamente un dato cargado?"
El eje X es ese año de arranque hipotético
El eje Y es la densidad resultante: cuánto de la matriz completa queda relleno si usás ese recorte.

Los últimos 2-3 años (2023, 2024, 2025) en general estan incompletos — los países tardan tiempo en reportar sus estadísticas al Banco Mundial, entonces esos años recientes tienen muchísimos huecos todavía (no es que falten datos "de verdad", es que todavía no llegaron/se cargaron).

Mientras el año de arranque está lejos del final (por ejemplo, 1990), esos años recientes tan vacíos son una porción chica del panel total (35 años), entonces no pesan mucho y la densidad se mantiene alta. Pero a medida que el año de arranque se acerca al final (2015, 2020), el panel se vuelve cada vez más corto, y esos años recientes vacíos pasan a ser una porción cada vez más grande del total — hasta que dominan el promedio y hacen caer la densidad.

# Sección 4: Análisis de dispersión entre los tres escenarios

Compara la dispersión de cobertura **entre países**  en tres escenarios: universo WDI completo (217), universo ONU (193) y universo ONU recortado a 1990-2024. Se usan dos métricas complementarias:

- **4.1 — Presencia acumulada:** `pct_indicadores_con_dato`, ya calculada más arriba para WDI y ONU. Cuenta un indicador como "cubierto" si el país tiene al menos un dato en cualquier año del rango considerado. No pondera por año: un solo dato aislado cuenta igual que una serie completa.
- **4.2 — Densidad ponderada por año:** % de celdas país-indicador-año efectivamente completas dentro del rango. A diferencia de la anterior, si un país reportó un indicador solo una vez en 65 años, pesa muy poco en esta métrica. Es la métrica correcta para evaluar el efecto real de recortar el panel a 1990-2024.

## 4.1 Dispersión de cobertura por país — indicador reportado al menos una vez

In [103]:
wdi_long_onu_9024 = wdi_long[
    (wdi_long["Country Code"].isin(paises_onu)) &
    (wdi_long["Year"] >= 1990) &
    (wdi_long["Year"] <= 2024)
]

indicadores_con_dato_9024 = (
    wdi_long_onu_9024.dropna(subset=["Value"])
    .groupby("Country Code")["Indicator Code"]
    .nunique()
    .rename("n_indicadores_con_dato_9024")
)

disponibilidad_pais_onu_9024 = (
    pd.DataFrame(index=paises_onu)
    .join(indicadores_con_dato_9024)
)
disponibilidad_pais_onu_9024["n_indicadores_con_dato_9024"] = disponibilidad_pais_onu_9024["n_indicadores_con_dato_9024"].fillna(0)
disponibilidad_pais_onu_9024["pct_indicadores_con_dato_9024"] = (
    disponibilidad_pais_onu_9024["n_indicadores_con_dato_9024"] / n_indicadores_total
)

disponibilidad_pais_onu_9024.head()


,n_indicadores_con_dato_9024,pct_indicadores_con_dato_9024
AFG,863,0.880612
AGO,937,0.956122
ALB,940,0.959184
AND,426,0.434694
ARE,805,0.821429


In [104]:
comparacion_dispersión_pais = pd.DataFrame({
    "Universo WDI (217 países)": disponibilidad_pais["pct_indicadores_con_dato"].describe(),
    "Universo ONU (193 países)": disponibilidad_pais_onu["pct_indicadores_con_dato"].describe(),
    "Universo ONU 1990-2024": disponibilidad_pais_onu_9024["pct_indicadores_con_dato_9024"].describe(),
})

comparacion_dispersión_pais.loc["IQR"] = comparacion_dispersión_pais.loc["75%"] - comparacion_dispersión_pais.loc["25%"]
comparacion_dispersión_pais.loc["CV"] = comparacion_dispersión_pais.loc["std"] / comparacion_dispersión_pais.loc["mean"]

comparacion_dispersión_pais
tablas_reporte.append(("dispersion_pais_presencia", comparacion_dispersión_pais))

In [105]:
df_dispersion_pais_presencia = pd.concat([
    pd.DataFrame({"pct_cobertura": disponibilidad_pais["pct_indicadores_con_dato"], "Universo": "WDI (217 países)"}),
    pd.DataFrame({"pct_cobertura": disponibilidad_pais_onu["pct_indicadores_con_dato"], "Universo": "ONU (193 países)"}),
    pd.DataFrame({"pct_cobertura": disponibilidad_pais_onu_9024["pct_indicadores_con_dato_9024"], "Universo": "ONU 1990-2024"}),
])

fig_dispersion_pais_presencia = px.box(
    df_dispersion_pais_presencia,
    x="Universo",
    y="pct_cobertura",
    points="outliers",
    labels={"pct_cobertura": "% de indicadores con al menos un dato"},
)

fig_dispersion_pais_presencia.update_layout(
    title="Dispersión de cobertura por país (presencia acumulada): WDI vs. ONU vs. ONU 1990-2024",
    xaxis_title="Universo de países considerado",
    yaxis_title="% de indicadores con al menos un dato",
)

figuras_reporte.append(("dispersion_pais_presencia_tres_escenarios", fig_dispersion_pais_presencia))
fig_dispersion_pais_presencia.show()


## 4.2 Dispersión de cobertura por país — densidad ponderada por año

Acá cada celda indicador-año cuenta individualmente: un país que reportó un indicador una sola vez en 65 años pesa muy distinto a uno que lo reportó todos los años. Por eso esta métrica sí puede mostrar cambios reales al recortar a 1990-2024, mientras que la de 4.1 estructuralmente no puede (recortar años nunca suma presencia acumulada, solo la mantiene o la resta).

In [106]:
def calcular_densidad_por_pais(datos_wdi, paises, indicadores, anio_inicio, anio_fin):
    sub = datos_wdi[
        datos_wdi["Country Code"].isin(paises) &
        datos_wdi["Indicator Code"].isin(indicadores) &
        (datos_wdi["Year"] >= anio_inicio) &
        (datos_wdi["Year"] <= anio_fin)
    ]
    n_celdas_por_pais = len(indicadores) * (anio_fin - anio_inicio + 1)
    celdas_con_dato_por_pais = (
        sub.dropna(subset=["Value"])
        .groupby("Country Code")
        .size()
    )
    densidad_pais = (
        pd.Series(index=paises, dtype=float)
        .fillna(0)
        .add(celdas_con_dato_por_pais.reindex(paises).fillna(0), fill_value=0)
        / n_celdas_por_pais
    )
    return densidad_pais

densidad_pais_wdi = calcular_densidad_por_pais(
    wdi_long, paises_soberanos, indicadores_candidatos, anio_min_disponible, anio_max_disponible
)
densidad_pais_onu = calcular_densidad_por_pais(
    wdi_long, paises_onu, indicadores_candidatos, anio_min_disponible, anio_max_disponible
)
densidad_pais_onu_9024 = calcular_densidad_por_pais(
    wdi_long, paises_onu, indicadores_candidatos, 1990, 2024
)

comparacion_densidad_pais = pd.DataFrame({
    "Universo WDI (217 países)": densidad_pais_wdi.describe(),
    "Universo ONU (193 países)": densidad_pais_onu.describe(),
    "Universo ONU 1990-2024": densidad_pais_onu_9024.describe(),
})

comparacion_densidad_pais.loc["IQR"] = comparacion_densidad_pais.loc["75%"] - comparacion_densidad_pais.loc["25%"]
comparacion_densidad_pais.loc["CV"] = comparacion_densidad_pais.loc["std"] / comparacion_densidad_pais.loc["mean"]

comparacion_densidad_pais
tablas_reporte.append(("dispersion_pais_densidad", comparacion_densidad_pais))

In [107]:
df_dispersion_pais_densidad = pd.concat([
    pd.DataFrame({"densidad": densidad_pais_wdi, "Universo": "WDI (217 países)"}),
    pd.DataFrame({"densidad": densidad_pais_onu, "Universo": "ONU (193 países)"}),
    pd.DataFrame({"densidad": densidad_pais_onu_9024, "Universo": "ONU 1990-2024"}),
])

fig_dispersion_pais_densidad = px.box(
    df_dispersion_pais_densidad,
    x="Universo",
    y="densidad",
    points="outliers",
    labels={"densidad": "% de celdas indicador-año con dato"},
)

fig_dispersion_pais_densidad.update_layout(
    title="Dispersión de cobertura por país (densidad ponderada por año): WDI vs. ONU vs. ONU 1990-2024",
    xaxis_title="Universo de países considerado",
    yaxis_title="% de celdas indicador-año con dato",
)

figuras_reporte.append(("dispersion_pais_densidad_tres_escenarios", fig_dispersion_pais_densidad))
fig_dispersion_pais_densidad.show()


# Bloque de exportación


In [108]:
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path_full = OUT_DIR / "disponibilidad_full.xlsx"

with pd.ExcelWriter(out_path_full) as writer:
    disponibilidad_indicador_full.to_excel(writer, sheet_name="por_indicador")
    disponibilidad_pais.to_excel(writer, sheet_name="por_pais")
    disponibilidad_anio.to_excel(writer, sheet_name="por_anio")

print(f"Exportado: {out_path_full}")


Exportado: data\processed\disponibilidad_full.xlsx


In [109]:
with pd.ExcelWriter(out_path_full, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
    comparacion_dispersión_pais.to_excel(writer, sheet_name="dispersion_pais_presencia")
    comparacion_densidad_pais.to_excel(writer, sheet_name="dispersion_pais_densidad")

print(f"Hojas de dispersión añadidas a: {out_path_full}")


Hojas de dispersión añadidas a: data\processed\disponibilidad_full.xlsx


In [110]:
with pd.ExcelWriter(out_path_full, mode="a", engine="openpyxl", if_sheet_exists="replace") as writer:
    df_wdi_sin_match_onu.to_excel(writer, sheet_name="match_wdi_no_onu", index=False)
    df_onu_sin_match_wdi.to_excel(writer, sheet_name="match_onu_sin_wdi", index=False)

print(f"Hojas de match añadidas a: {out_path_full}")


Hojas de match añadidas a: data\processed\disponibilidad_full.xlsx


In [111]:
import re
import nbformat

NOTEBOOK_PATH = "02_disponibilidad_datos_wdi.ipynb"  # ajustar si el nombre/ruta difiere

figuras_por_nombre = dict(figuras_reporte)
tablas_por_nombre = dict(tablas_reporte)

def markdown_a_html(texto):
    """Conversor liviano de Markdown a HTML (headers, negrita, código inline, listas, párrafos)."""
    lineas = texto.split("\n")
    html = []
    en_lista = False
    for linea in lineas:
        l = linea.rstrip()
        if l.startswith("### "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h3>{l[4:]}</h3>")
        elif l.startswith("## "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h2>{l[3:]}</h2>")
        elif l.startswith("# "):
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<h1>{l[2:]}</h1>")
        elif l.startswith("- "):
            if not en_lista:
                html.append("<ul>"); en_lista = True
            html.append(f"<li>{l[2:]}</li>")
        elif l.strip() == "":
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append("")
        else:
            if en_lista:
                html.append("</ul>"); en_lista = False
            html.append(f"<p>{l}</p>")
    if en_lista:
        html.append("</ul>")
    texto_html = "\n".join(html)
    texto_html = re.sub(r"\*\*(.+?)\*\*", r"<strong>\1</strong>", texto_html)
    texto_html = re.sub(r"`(.+?)`", r"<code>\1</code>", texto_html)
    return texto_html

nb_en_disco = nbformat.read(NOTEBOOK_PATH, as_version=4)

partes_html = []
nombres_incrustados = set()

for cell in nb_en_disco.cells:
    if cell.cell_type == "markdown":
        partes_html.append(markdown_a_html(cell.source))
    elif cell.cell_type == "code":
        for nombre in re.findall(r'figuras_reporte\.append\(\(\s*"([^"]+)"', cell.source):
            if nombre in figuras_por_nombre:
                partes_html.append(f"<h4>Figura: {nombre}</h4>")
                partes_html.append(figuras_por_nombre[nombre].to_html(full_html=False, include_plotlyjs="cdn"))
                nombres_incrustados.add(("figura", nombre))
        for nombre in re.findall(r'tablas_reporte\.append\(\(\s*"([^"]+)"', cell.source):
            if nombre in tablas_por_nombre:
                partes_html.append(f"<h4>Tabla: {nombre}</h4>")
                partes_html.append(tablas_por_nombre[nombre].to_html())
                nombres_incrustados.add(("tabla", nombre))

html_path = OUT_DIR / "reporte_disponibilidad_wdi.html"
with open(html_path, "w", encoding="utf-8") as f:
    f.write("<html><head><meta charset='utf-8'><title>Reporte de disponibilidad WDI</title></head><body>\n")
    f.write("\n".join(partes_html))
    f.write("\n</body></html>")

figuras_faltantes = set(figuras_por_nombre) - {n for t, n in nombres_incrustados if t == "figura"}
tablas_faltantes = set(tablas_por_nombre) - {n for t, n in nombres_incrustados if t == "tabla"}

print(f"Exportado: {html_path}")
print(f"Figuras incrustadas: {len(figuras_por_nombre) - len(figuras_faltantes)} / {len(figuras_por_nombre)}")
print(f"Tablas incrustadas: {len(tablas_por_nombre) - len(tablas_faltantes)} / {len(tablas_por_nombre)}")
if figuras_faltantes:
    print(f"⚠️ Figuras en memoria pero no encontradas en el .ipynb guardado: {figuras_faltantes}")
if tablas_faltantes:
    print(f"⚠️ Tablas en memoria pero no encontradas en el .ipynb guardado: {tablas_faltantes}")

Exportado: data\processed\reporte_disponibilidad_wdi.html
Figuras incrustadas: 20 / 20
Tablas incrustadas: 2 / 2
